# 02 — 7500 hotels from Airbnb, Booking and Hotels.com

**Étape du pipeline** : *Raw data → Data transformation*.

Source : [Kaggle, mykhailozub](https://www.kaggle.com/datasets/mykhailozub/500-hotels-from-airbnb-booking-and-hotelscom), un JSON par ville (collecte 2023). On garde Londres, Berlin, Paris et Rome. Madrid est exclue car elle n'existe pas côté Airbnb.

Sortie : `data/interim/hotels.parquet`, au schéma commun.

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prep").is_dir())
sys.path.insert(0, str(RACINE))
RAW, INTERIM, PROCESSED = (RACINE / "data" / d for d in ("raw", "interim", "processed"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

import json
from src.prep import hotels, schema
from src.prep.common import USD_PAR_EUR

## 1. Structure du JSON

Chaque fichier contient **trois listes**, une par site.

In [ ]:
for f in sorted((RAW / "hotels").glob("*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    print(f.name, {k: len(v) for k, v in d.items()})

In [ ]:
paris = json.loads((RAW / "hotels" / "Paris.json").read_text(encoding="utf-8"))
for liste in paris:
    print(f"--- {liste}")
    print(json.dumps(paris[liste][0], ensure_ascii=False, indent=1)[:900])

## 2. Quelle liste garder ? (question ouverte n° 4)

| Champ utile | `bookingHotels` | `hotelsComHotels` | `airbnbHotels` |
|---|---|---|---|
| prix | oui (US$, taxes souvent absentes) | oui ($, avec et sans taxes) | oui ($, **par nuit**) |
| note | /10 + nb d'avis | /10 + nb d'avis | /5 |
| étoiles | oui | non | — |
| quartier | **oui** | non (presque toujours la ville seule) | non |
| distance au centre | **oui** | non | non |
| accès métro | **oui** | non | non |

**Décision** : on garde **Booking seul**. Hotels.com n'a ni quartier, ni distance, ni métro, donc il est inutilisable pour notre problématique. La liste `airbnbHotels` ne sert qu'à un contrôle de cohérence des prix (section 3).

In [ ]:
hc = hotels.charger_brut(RAW / "hotels", "hotelsComHotels")
hc.groupby("ville_brut")["location"].agg(lambda s: s.value_counts().head(3).to_dict())

## 3. Le prix Booking est-il par nuit ? (question ouverte n° 1)

In [ ]:
brut = hotels.charger_brut(RAW / "hotels")
ab = hotels.charger_brut(RAW / "hotels", "airbnbHotels")
print("airbnbHotels, champ period :", ab["period"].value_counts().to_dict())
pd.DataFrame({
    "Booking (médiane $)": brut.groupby("ville_brut")["price"].median(),
    "Hotels.com (médiane $)": hc.groupby("ville_brut")["price"].median(),
    "Airbnb du même dataset, par nuit ($)": ab.groupby("ville_brut")["price"].median(),
})

In [ ]:
brut.pivot_table(index="stars", columns="ville_brut", values="price", aggfunc="median").round(0)

**Constats**

- Pour Londres, Berlin et Paris, les médianes Booking sont proches de celles de Hotels.com et plausibles pour une nuit. On les traite comme des **prix par nuit**.
- **Rome est une anomalie** : à nombre d'étoiles égal, les prix y sont 2 à 5 fois plus élevés qu'ailleurs. Le dataset ne compte que 208 hôtels pour Rome, contre 500 ailleurs. Le rapport n'est pas constant (×4 pour les 3★, ×2 pour les 5★), donc on ne peut pas le corriger par un nombre de nuits.
  → **Décision de l'équipe** : les hôtels de Rome sont gardés, y compris dans les comparaisons de prix. Ils sont marqués `prix_comparable = False` pour que l'interface affiche un avertissement.
- Les dates de séjour ne figurent pas dans les données Booking. Côté `airbnbHotels`, les sous-titres montrent des séjours de mars à décembre 2023.

## 4. Taxes

In [ ]:
brut.groupby("ville_brut").apply(lambda g: pd.Series({
    "taxes renseignées": g["taxesAndCharges"].notna().mean(),
    "taxes / prix (médiane)": (g["taxesAndCharges"] / g["price"]).median(),
})).round(3)

**Constat** : `taxesAndCharges` est absent pour 95 % des hôtels de Londres et 58 % de ceux de Berlin. Quand il est renseigné à Londres, il est incohérent (de 2 % à plus de 1000 % du prix). Ailleurs, il vaut 2 à 5 % du prix (taxe de séjour).

**Décision** : on retient le **prix affiché seul** (`price`), disponible partout et homogène entre les villes. Il faut savoir qu'il sous-estime légèrement le coût réel (taxe de séjour).

Conversion : 1 € = 1,0813 $ (moyenne annuelle 2023 de la BCE).

## 5. Distance au centre : unités mélangées

In [ ]:
d = brut["distanceFromCenter"]
print("valeurs >= 25 :", sorted(d[d >= 25].unique()))
print("plus petite valeur < 25 :", d[d < 25].min())

**Constat** : les valeurs comprises entre 100 et 500, par pas de 50, sont des **mètres** (« 450 m from centre ») lus comme des km. La plus petite valeur en km est 0,5. **Règle** : une valeur ≥ 100 est convertie de m en km. La valeur 26,9 km (Berlin) est une vraie distance, on la garde.

## 6. Valeurs manquantes et doublons

In [ ]:
print("note absente (« No rating ») :", brut["score"].isna().sum())
print("étoiles absentes :", brut["stars"].isna().sum(), f"({brut['stars'].isna().mean():.0%})")
print("doublons (même lien, même ville) :", brut.duplicated(["ville_brut", "link"]).sum())
print("logements entiers proposés sur Booking :", brut["highlights"].str.contains(r"\bEntire ", regex=True).sum())

- Les **étoiles** manquent pour 25 % des établissements (classement non déclaré). Elles restent vides et ne sont pas imputées.
- Booking propose aussi des **logements entiers**, repérés par « Entire apartment/villa » dans les `highlights`. Leur `type` est `Logement entier`, mais leur `source` reste `Hôtel`. On peut ainsi les filtrer côté D3.

## 7. Quartiers

In [ ]:
brut.assign(q=[hotels.normaliser_quartier(l, hotels.FICHIERS[v]) for l, v in zip(brut.location, brut.ville_brut)]) \
    .groupby("ville_brut")["q"].agg(["nunique", lambda s: s.isna().mean().round(3)])

Le champ `location` donne le quartier pour presque tous les hôtels. Le découpage n'est pas homogène :
- **Paris** : arrondissements, propres ;
- **Londres** : surtout des boroughs, plus quelques quartiers (Bayswater, Chiswick…) ;
- **Berlin** : surtout des Bezirke, plus quelques Ortsteile (Prenzlauer Berg…) ;
- **Rome** : des quartiers touristiques (Spagna, Trevi…), sans lien avec les municipi.

Le rattachement aux contours passe par la table de correspondance `data/external/correspondance_quartiers.csv` (notebook 03).

## 8. Transformation vers le schéma commun

In [ ]:
df, liens = hotels.transformer_avec_liens(hotels.dedoublonner(brut))
df = schema.valider(df)
print(df.shape)
df.head()

In [ ]:
df.groupby(["ville", "tranche_dist"], observed=True).size().unstack()

## 9. Export intermédiaire

In [ ]:
INTERIM.mkdir(parents=True, exist_ok=True)
df.to_parquet(INTERIM / "hotels.parquet", index=False)
print(INTERIM / "hotels.parquet", len(df))
# Table id -> lien Booking : clé de jointure avec le géocodage (notebook 04), utilisée au notebook 05.
liens.to_parquet(INTERIM / "hotels_liens.parquet", index=False)